In [ ]:
import pandas as pd
import os
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, FilterCatalog, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem import Draw
from rdkit.Chem import QED
import re
from collections import defaultdict
from IPython.display import display
import math

# Generating compound library

The molecular structures of the compounds to be investigated as therapeutic agonists are created and saved as .mol files. 
- *Typical file naming: CompoundName.mol*
These structures are extracted for further analysis.

In [ ]:
# Import novel compound library

def load_mols_from_folder(folder_path):
    mol_list = []
    
    filenames = sorted([f for f in os.listdir(folder_path) if f.endswith('.mol')])
    
    for filename in filenames:
        file_path = os.path.join(folder_path, filename)
        
        mol = Chem.MolFromMolFile(file_path)
        
        if mol is not None:
            mol.SetProp("_Name", filename)
            mol_list.append(mol)
        else:
            print(f"Warning: Could not load {filename}")
            
    return mol_list

my_mols = load_mols_from_folder()
print(f"Successfully loaded {len(my_mols)} molecules.")

In [ ]:
# Group compounds into classes

classes = defaultdict(list)

folder_path = ""
filenames = sorted([f for f in os.listdir(folder_path) if f.endswith('.mol')])


for filename in filenames:
    file_path = os.path.join(folder_path, filename)
    mol = Chem.MolFromMolFile(file_path)
    
    if mol is not None:
        base_name = os.path.splitext(filename)[0]
        mol.SetProp("_Name", base_name)

        match = re.search(r'([A-Za-z]+)', base_name)
        if match:
                group_key = match.group(1)
                classes[group_key].append(mol)
        else:
            print(f"Skipping {filename}: Could not parse molecule.")


print(f"Classes: {list(classes.keys())}")

for class_name, mols in classes.items():
    print(f"--- {class_name} ({len(mols)} molecules) ---")

# Obtain molecular structures

Transform the extracted files into 2D renderings of their molecular structures using RDKit.Chem

*This script was written for files whereby the compound name comtained the chemical class that the molecule was classified into. Adjust as necessary.*

In [ ]:
# Draw compound library - molecules in different classes

output_dir = ""
os.makedirs(output_dir, exist_ok=True)

for class_name, mols in classes.items():
    print(f"--- {class_name} ({len(mols)} molecules) ---")
    
    labels = [m.GetProp("_Name") for m in mols]
    
    # Generate the grid
    mols_2D_structures = Draw.MolsToGridImage(
        mols, 
        molsPerRow=4, 
        legends=labels,
        returnPNG=True
    )
    
    display(mols_2D_structures)

    filename = os.path.join(output_dir, f"{class_name}_2D.png")
    with open(filename, "wb") as f:
        f.write(mols_2D_structures.data)
        
    print(f"Saved: {filename}")

all_mols = []
all_labels = []

for class_name, mols in classes.items():
    for m in mols:
        all_mols.append(m)
        name = m.GetProp("_Name") if m.HasProp("_Name") else "Unknown"
        all_labels.append(f"{name}")

combined_filename = os.path.join(output_dir, "All_Compounds_2D.png")

full_grid = Draw.MolsToGridImage(
    all_mols, 
    molsPerRow=6, 
    legends=all_labels,
    returnPNG=True
)

with open(combined_filename, "wb") as f:
    f.write(full_grid.data)

display(full_grid)
print(f"Saved all {len(all_mols)} compounds to: {combined_filename}")


# Get molecule descriptives

The following are essential parameters, but additional descriptives may be added as needed:
- Molecular weight
- LogP 
- Number of hydrogen bond donors & acceptors
- Pan-assay Interference (PAINS) motifs
- Topological surface area (TPSA)
- Number of rotatable bonds
- Number of Lipinski's Rule of 5 violations

In [ ]:
results = []

params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)

for class_name, mols in classes.items():
    for mol in mols:
        if mol is None: continue
        
        mw = Descriptors.MolWt(mol)
        clogp = Crippen.MolLogP(mol)
        tpsa = rdMolDescriptors.CalcTPSA(mol)
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        has_pains = pains_catalog.HasMatch(mol)
        pains_status = "FAIL" if has_pains else "PASS"
        Lipinski = (mw <= 500 and clogp <= 5 and hbd <= 5 and hba <= 10)
        Lipinski_status = "PASS" if Lipinski else "FAIL"
        mw_fail  = mw > 500
        logp_fail = clogp > 5
        hbd_fail = hbd > 5
        hba_fail = hba > 10
        lipinski_score = sum([mw_fail, logp_fail, hbd_fail, hba_fail])
        
        results.append({
            "Class": class_name,
            "ID": mol.GetProp("_Name"),
            "SMILES": Chem.MolToSmiles(mol),
            "MW": round(mw, 2),
            "LogP": clogp,
            "TPSA": round(tpsa, 2),
            "HBD": hbd,
            "HBA": hba,
            "Rot_Bonds": rot_bonds,
            "PAINS": pains_status,
            "Lipinski_Pass": Lipinski_status,
            "Lipinksi_Violations": lipinski_score
        })

df = pd.DataFrame(results)
df = df.sort_values(by=["Class", "ID"])

output_file = ""
df.to_excel(output_file, index=False)

print(f"Excel table generated successfully: {output_file}")

# Filter out poor candidates

Based on the molecular descriptives of the compound library, filter out poor candidates.

*In this script, drug-likeliness was evaluated based on PAINS and Lipinski's rule of 5. Adjust these parameters as needed.*

In [ ]:
# No PAINS motifs & less than or equal to 1 Ro5 violations

Criteria_Stage1 = (df["Lipinksi_Violations"] <= 1) & (df["PAINS"] == 'PASS')

count = Criteria_Stage1.sum()
print(count)

Compounds_Stage2 = df[Criteria_Stage1]['ID']
print(Compounds_Stage2.to_list())

columns_to_export = ['Class', 'ID', 'MW', 'LogP', 'TPSA', 'Rot_Bonds', 'HBD', 'HBA', 'Lipinksi_Violations']
df_stage2 = df.loc[Criteria_Stage1, columns_to_export]

output_file2 = ""
df_stage2.to_excel(output_file2, index=False)

print(f"Exported {len(df_stage2)} compounds to Excel.")


